# Notebook 03 — Full 500-Patient 3D Patch Extraction

This is the final patient-level patch-extraction notebook for the labelled RibFrac cohort.

**Labelled source pool**
- Training Part 1: 300 patients
- Training Part 2: 120 patients
- Labelled validation: 80 patients
- Total: **500 labelled patients**

The notebook creates a new patient-level split before patch extraction, then extracts patches patient-by-patient to limit RAM usage.

**Important:** the generated `test/` split is an **internal held-out labelled test set**, not the official RibFrac challenge test set.


In [ ]:
# Cell 1 — Imports and reproducibility
import gc
import hashlib
import json
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("Libraries imported successfully.")
print("Random seed:", SEED)


In [ ]:
# Cell 2 — Dataset paths

PART1_DATASET_ROOT = Path(
    "/kaggle/input/datasets/waboke/ribfrac-processed-dev"
)

# IMPORTANT: verify these two Kaggle Dataset slugs before running
# the full extraction.
PART2_DATASET_ROOT = Path(
    "/kaggle/input/datasets/waboke/ribfrac-processed-part2"
)

VALIDATION_DATASET_ROOT = Path(
    "/kaggle/input/datasets/waboke/ribfrac-processed-validation"
)

PATCH_ROOT = Path(
    "/kaggle/working/ribfrac-patch-dataset-final"
)

PATCH_ROOT.mkdir(parents=True, exist_ok=True)

print("Part 1:", PART1_DATASET_ROOT)
print("Part 2:", PART2_DATASET_ROOT)
print("Labelled validation:", VALIDATION_DATASET_ROOT)
print("Patch output:", PATCH_ROOT)


In [ ]:
# Cell 3 — Locate a compatible preprocessed source

def locate_source(root, name):
    candidates = [
        {
            "image_dir": root / "ribfrac_preprocessed" / "images",
            "label_dir": root / "ribfrac_preprocessed" / "labels",
            "info_path": root / "ribfrac_metadata" / "ribfrac-train-info-1.csv",
        },
        {
            "image_dir": root / "images",
            "label_dir": root / "labels",
            "info_path": root / "ribfrac-train-info-1.csv",
        },
    ]

    for c in candidates:
        if (
            c["image_dir"].exists()
            and c["label_dir"].exists()
            and c["info_path"].exists()
        ):
            c["name"] = name
            return c

    raise FileNotFoundError(
        f"Compatible preprocessed source not found for {name}: {root}"
    )

PART1 = locate_source(PART1_DATASET_ROOT, "part1")
PART2 = locate_source(PART2_DATASET_ROOT, "part2")
VAL_SOURCE = locate_source(VALIDATION_DATASET_ROOT, "validation")

for source in (PART1, PART2, VAL_SOURCE):
    print(
        source["name"],
        "images =", source["image_dir"],
        "labels =", source["label_dir"],
        "metadata =", source["info_path"],
    )


In [ ]:
# Cell 4 — Verify expected patient counts

def patient_ids_from_images(source):
    return {
        p.name[:-len("-image.npy")]
        for p in source["image_dir"].glob("*-image.npy")
    }

part1_patients = patient_ids_from_images(PART1)
part2_patients = patient_ids_from_images(PART2)
val_patients = patient_ids_from_images(VAL_SOURCE)

print("Part 1:", len(part1_patients))
print("Part 2:", len(part2_patients))
print("Validation:", len(val_patients))

assert len(part1_patients) == 300
assert len(part2_patients) == 120
assert len(val_patients) == 80

assert not (part1_patients & part2_patients)
assert not (part1_patients & val_patients)
assert not (part2_patients & val_patients)

all_patients = sorted(
    part1_patients | part2_patients | val_patients
)

assert len(all_patients) == 500

print("500 labelled patients verified: PASS")


In [ ]:
# Cell 5 — Load combined annotation metadata

REQUIRED_COLUMNS = {"public_id", "label_id", "label_code"}

def load_metadata(source):
    df = pd.read_csv(source["info_path"]).copy()
    missing = REQUIRED_COLUMNS - set(df.columns)

    if missing:
        raise ValueError(
            f"{source['name']} metadata missing: {missing}"
        )

    df["public_id"] = df["public_id"].astype(str)
    df["label_id"] = df["label_id"].astype(int)
    df["label_code"] = df["label_code"].astype(int)
    df["source"] = source["name"]
    return df

info_df = pd.concat(
    [
        load_metadata(PART1),
        load_metadata(PART2),
        load_metadata(VAL_SOURCE),
    ],
    ignore_index=True,
)

print("Metadata rows:", len(info_df))
print("Unique patients:", info_df["public_id"].nunique())

assert info_df["public_id"].nunique() == 500


## Final patient-level split

The 500 labelled patients are split **before patch extraction**:

- 350 training patients
- 75 validation patients
- 75 internal held-out test patients

`random_state=42` is fixed for reproducibility.


In [ ]:
# Cell 6 — Create final patient-level split

train_patients, temp_patients = train_test_split(
    all_patients,
    test_size=0.30,
    random_state=SEED,
    shuffle=True,
)

validation_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    random_state=SEED,
    shuffle=True,
)

train_patients = sorted(train_patients.tolist())
validation_patients = sorted(validation_patients.tolist())
test_patients = sorted(test_patients.tolist())

assert len(train_patients) == 350
assert len(validation_patients) == 75
assert len(test_patients) == 75

train_set = set(train_patients)
validation_set = set(validation_patients)
test_set = set(test_patients)

assert train_set.isdisjoint(validation_set)
assert train_set.isdisjoint(test_set)
assert validation_set.isdisjoint(test_set)

print("Train:", len(train_patients))
print("Validation:", len(validation_patients))
print("Internal test:", len(test_patients))
print("Patient-level split integrity: PASS")


In [ ]:
# Cell 7 — Save split and source registry

def source_for_patient(pid):
    if pid in part1_patients:
        return "part1"
    if pid in part2_patients:
        return "part2"
    if pid in val_patients:
        return "validation"
    raise ValueError(pid)

rows = []

for pid in train_patients:
    rows.append({"public_id": pid, "split": "train", "source": source_for_patient(pid)})

for pid in validation_patients:
    rows.append({"public_id": pid, "split": "validation", "source": source_for_patient(pid)})

for pid in test_patients:
    rows.append({"public_id": pid, "split": "test", "source": source_for_patient(pid)})

patient_registry = pd.DataFrame(rows)

patient_registry.to_csv(
    PATCH_ROOT / "patient_split_500.csv",
    index=False,
)

patient_registry.to_csv(
    PATCH_ROOT / "patient_source_registry.csv",
    index=False,
)

print(patient_registry.groupby(["source", "split"]).size())


In [ ]:
# Cell 8 — Patch configuration

PATCH_SIZE = (64, 64, 64)
STRIDE = (32, 32, 32)

NEGATIVE_TO_POSITIVE_RATIO = 1.0
MIN_FRACTURE_VOXELS = 1
AMBIGUOUS_LABEL_CODE = -1

CLASS_NAMES = {
    0: "background",
    1: "displaced",
    2: "non-displaced",
    3: "buckle",
    4: "segmental",
}

LABEL_CODE_TO_CLASS = {
    1: 1,
    2: 2,
    3: 3,
    4: 4,
}

print("Patch size:", PATCH_SIZE)
print("Stride:", STRIDE)
print("Negative/positive ratio:", NEGATIVE_TO_POSITIVE_RATIO)
print("Classes:", CLASS_NAMES)


In [ ]:
# Cell 9 — Source lookup and metadata lookup

SOURCE_MAP = {
    "part1": PART1,
    "part2": PART2,
    "validation": VAL_SOURCE,
}

def get_source(pid):
    source_name = patient_registry.loc[
        patient_registry["public_id"] == pid,
        "source"
    ].iloc[0]
    return SOURCE_MAP[source_name]

def get_label_map(pid):
    rows = info_df[info_df["public_id"] == pid]
    return {
        int(row.label_id): int(row.label_code)
        for row in rows.itertuples()
    }


In [ ]:
# Cell 10 — Patient loader

def load_patient(pid):
    source = get_source(pid)

    ct_path = source["image_dir"] / f"{pid}-image.npy"
    label_path = source["label_dir"] / f"{pid}-label.npy"

    if not ct_path.exists():
        raise FileNotFoundError(ct_path)
    if not label_path.exists():
        raise FileNotFoundError(label_path)

    ct = np.load(ct_path, mmap_mode="r")
    label = np.load(label_path, mmap_mode="r")

    if ct.shape != label.shape:
        raise ValueError(
            f"{pid}: CT/label shape mismatch: {ct.shape} vs {label.shape}"
        )

    if ct.ndim != 3:
        raise ValueError(
            f"{pid}: expected 3D volumes, got {ct.shape}"
        )

    return ct, label


In [ ]:
# Cell 11 — Patch coordinate generation

def generate_patch_starts(volume_shape):
    starts = []

    for z in range(
        0,
        volume_shape[0] - PATCH_SIZE[0] + 1,
        STRIDE[0],
    ):
        for y in range(
            0,
            volume_shape[1] - PATCH_SIZE[1] + 1,
            STRIDE[1],
        ):
            for x in range(
                0,
                volume_shape[2] - PATCH_SIZE[2] + 1,
                STRIDE[2],
            ):
                starts.append((z, y, x))

    return starts


In [ ]:
# Cell 12 — Build valid and ambiguous masks

def build_masks(label, label_map):
    valid = np.zeros(label.shape, dtype=np.uint8)
    ambiguous = np.zeros(label.shape, dtype=np.uint8)

    for label_id, label_code in label_map.items():
        if label_code in (1, 2, 3, 4):
            valid[label == label_id] = 1
        elif label_code == AMBIGUOUS_LABEL_CODE:
            ambiguous[label == label_id] = 1

    return valid, ambiguous


In [ ]:
# Cell 13 — Determine patch class

def patch_class(label_patch, label_map):
    counts = Counter()
    ambiguous = False

    for label_id in np.unique(label_patch):
        label_id = int(label_id)

        if label_id == 0:
            continue

        label_code = label_map.get(label_id)

        if label_code is None:
            ambiguous = True
            continue

        if label_code == AMBIGUOUS_LABEL_CODE:
            ambiguous = True
            continue

        if label_code in LABEL_CODE_TO_CLASS:
            class_id = LABEL_CODE_TO_CLASS[label_code]
            counts[class_id] += int(
                np.sum(label_patch == label_id)
            )

    if ambiguous:
        return None, "ambiguous"

    total = sum(counts.values())

    if total < MIN_FRACTURE_VOXELS:
        return 0, "background"

    maximum = max(counts.values())
    dominant = [
        cls for cls, count in counts.items()
        if count == maximum
    ]

    if len(dominant) != 1:
        return None, "mixed_class_tie"

    return dominant[0], "positive"


## Patient extraction policy

For each candidate patch:

1. **Ambiguous** → exclude.
2. **Multiple valid classes tied for dominance** → exclude.
3. **No valid fracture** → clean background candidate.
4. **Valid fracture present** → positive patch with the unique dominant class.

Negative patches are sampled at a maximum 1:1 ratio relative to positive patches.

The full cohort is processed one patient at a time.


In [ ]:
# Cell 14 — Extract and save one patient

def extract_patient(pid, split_name):
    ct, label = load_patient(pid)
    label_map = get_label_map(pid)

    valid_mask, ambiguous_mask = build_masks(
        label,
        label_map,
    )

    starts = generate_patch_starts(ct.shape)

    positives = []
    negatives = []
    excluded = Counter()

    for start in starts:
        z, y, x = start

        z2 = z + PATCH_SIZE[0]
        y2 = y + PATCH_SIZE[1]
        x2 = x + PATCH_SIZE[2]

        label_patch = np.asarray(
            label[z:z2, y:y2, x:x2]
        )

        cls, reason = patch_class(
            label_patch,
            label_map,
        )

        if reason == "ambiguous":
            excluded["ambiguous"] += 1
            continue

        if reason == "mixed_class_tie":
            excluded["mixed_class_tie"] += 1
            continue

        seg_patch = np.asarray(
            valid_mask[z:z2, y:y2, x:x2]
        )

        if np.any(
            ambiguous_mask[z:z2, y:y2, x:x2]
        ):
            excluded["ambiguous"] += 1
            continue

        if cls == 0:
            if np.any(seg_patch):
                raise RuntimeError(
                    "Negative patch contains fracture voxels."
                )
            negatives.append((start, 0))

        elif cls in (1, 2, 3, 4):
            if not np.any(seg_patch):
                raise RuntimeError(
                    "Positive patch contains no fracture voxels."
                )
            positives.append((start, cls))

    # Stable per-patient seed: Python hash() is intentionally avoided.
    digest = hashlib.sha256(
        str(pid).encode("utf-8")
    ).hexdigest()
    patient_seed = SEED + int(digest[:8], 16)

    rng = random.Random(patient_seed)

    rng.shuffle(positives)
    rng.shuffle(negatives)

    max_negative = int(
        len(positives) * NEGATIVE_TO_POSITIVE_RATIO
    )

    selected = positives + negatives[:max_negative]
    rng.shuffle(selected)

    out = PATCH_ROOT / split_name / str(pid)
    out.mkdir(parents=True, exist_ok=True)

    X = np.empty(
        (len(selected), *PATCH_SIZE, 1),
        dtype=np.float32,
    )

    Y_seg = np.empty(
        (len(selected), *PATCH_SIZE, 1),
        dtype=np.uint8,
    )

    Y_class = np.empty(
        (len(selected),),
        dtype=np.int64,
    )

    manifest = []

    for i, (start, cls) in enumerate(selected):
        z, y, x = start

        ct_patch = np.asarray(
            ct[z:z+PATCH_SIZE[0],
               y:y+PATCH_SIZE[1],
               x:x+PATCH_SIZE[2]],
            dtype=np.float32,
        )

        seg_patch = np.asarray(
            valid_mask[z:z+PATCH_SIZE[0],
                       y:y+PATCH_SIZE[1],
                       x:x+PATCH_SIZE[2]],
            dtype=np.uint8,
        )

        X[i, ..., 0] = ct_patch
        Y_seg[i, ..., 0] = seg_patch
        Y_class[i] = cls

        manifest.append({
            "patient_id": pid,
            "split": split_name,
            "patch_index": i,
            "start_z": z,
            "start_y": y,
            "start_x": x,
            "classification": cls,
            "classification_name": CLASS_NAMES[cls],
            "fracture_voxels": int(seg_patch.sum()),
        })

    np.save(out / "X.npy", X)
    np.save(out / "Y_seg.npy", Y_seg)
    np.save(out / "Y_class.npy", Y_class)

    pd.DataFrame(manifest).to_csv(
        out / "manifest.csv",
        index=False,
    )

    stats = {
        "patient_id": pid,
        "split": split_name,
        "candidate_patches": len(starts),
        "positive_candidates": len(positives),
        "negative_candidates": len(negatives),
        "selected_positive": len(positives),
        "selected_negative": min(
            len(negatives),
            max_negative,
        ),
        "selected_total": len(selected),
        "excluded_ambiguous": excluded["ambiguous"],
        "excluded_mixed_class_tie": excluded["mixed_class_tie"],
    }

    del X, Y_seg, Y_class
    del ct, label, valid_mask, ambiguous_mask
    gc.collect()

    return stats


In [ ]:
# Cell 15 — Smoke test one patient

SMOKE_TEST_PATIENT = train_patients[0]

print("Smoke-test patient:", SMOKE_TEST_PATIENT)

smoke_stats = extract_patient(
    SMOKE_TEST_PATIENT,
    "train",
)

print("\nSmoke-test statistics:")
print(smoke_stats)

smoke_dir = (
    PATCH_ROOT
    / "train"
    / SMOKE_TEST_PATIENT
)

X = np.load(
    smoke_dir / "X.npy",
    mmap_mode="r",
)
Y_seg = np.load(
    smoke_dir / "Y_seg.npy",
    mmap_mode="r",
)
Y_class = np.load(
    smoke_dir / "Y_class.npy",
    mmap_mode="r",
)

assert X.ndim == 5
assert Y_seg.ndim == 5
assert Y_class.ndim == 1

assert X.shape[1:] == (*PATCH_SIZE, 1)
assert Y_seg.shape[1:] == (*PATCH_SIZE, 1)
assert X.shape[0] == Y_seg.shape[0] == Y_class.shape[0]

assert set(np.unique(Y_seg)).issubset({0, 1})
assert set(np.unique(Y_class)).issubset(set(CLASS_NAMES))

print("\nSmoke-test validation: PASS")


In [ ]:
# Cell 16 — Full extraction switch

RUN_FULL_EXTRACTION = False

print(
    "RUN_FULL_EXTRACTION =",
    RUN_FULL_EXTRACTION,
)

if not RUN_FULL_EXTRACTION:
    print(
        "Full extraction is disabled. "
        "Set RUN_FULL_EXTRACTION = True only after Cell 15 passes."
    )


In [ ]:
# Cell 17 — Process all 500 patients

split_patients = {
    "train": train_patients,
    "validation": validation_patients,
    "test": test_patients,
}

all_statistics = []

if RUN_FULL_EXTRACTION:

    for split_name, patients in split_patients.items():

        print("\n" + "=" * 80)
        print(
            f"PROCESSING {split_name.upper()}: "
            f"{len(patients)} patients"
        )
        print("=" * 80)

        for index, pid in enumerate(
            patients,
            start=1,
        ):

            print(
                f"[{index}/{len(patients)}] {pid}"
            )

            try:
                stats = extract_patient(
                    pid,
                    split_name,
                )
                all_statistics.append(stats)

            except Exception as exc:

                print(
                    f"ERROR: {pid}: {exc}"
                )

                all_statistics.append({
                    "patient_id": pid,
                    "split": split_name,
                    "error": str(exc),
                })

            gc.collect()

    extraction_df = pd.DataFrame(
        all_statistics
    )

    summary_path = (
        PATCH_ROOT
        / "manifests"
        / "full_extraction_summary.csv"
    )

    summary_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    extraction_df.to_csv(
        summary_path,
        index=False,
    )

    print("\nSaved:", summary_path)

else:
    print("Full extraction not executed.")


In [ ]:
# Cell 18 — Validate all 500 patient directories

def validate_dataset():
    errors = []

    for split_name, patients in split_patients.items():

        split_dir = PATCH_ROOT / split_name

        if not split_dir.exists():
            errors.append(
                f"Missing split directory: {split_name}"
            )
            continue

        found = {
            p.name
            for p in split_dir.iterdir()
            if p.is_dir()
        }

        expected = set(patients)

        if found != expected:
            errors.append(
                f"{split_name}: expected {len(expected)} "
                f"patient dirs, found {len(found)}"
            )

        for pid in patients:

            patient_dir = split_dir / pid

            required = [
                patient_dir / "X.npy",
                patient_dir / "Y_seg.npy",
                patient_dir / "Y_class.npy",
                patient_dir / "manifest.csv",
            ]

            for path in required:
                if not path.exists():
                    errors.append(
                        f"Missing: {path}"
                    )

            if not patient_dir.exists():
                continue

            X = np.load(
                patient_dir / "X.npy",
                mmap_mode="r",
            )
            Y_seg = np.load(
                patient_dir / "Y_seg.npy",
                mmap_mode="r",
            )
            Y_class = np.load(
                patient_dir / "Y_class.npy",
                mmap_mode="r",
            )

            if X.shape[0] != Y_seg.shape[0]:
                errors.append(
                    f"{pid}: X/Y_seg mismatch"
                )

            if X.shape[0] != Y_class.shape[0]:
                errors.append(
                    f"{pid}: X/Y_class mismatch"
                )

            if X.shape[1:] != (*PATCH_SIZE, 1):
                errors.append(
                    f"{pid}: invalid X shape {X.shape}"
                )

            if Y_seg.shape[1:] != (*PATCH_SIZE, 1):
                errors.append(
                    f"{pid}: invalid Y_seg shape {Y_seg.shape}"
                )

            if not set(np.unique(Y_seg)).issubset({0, 1}):
                errors.append(
                    f"{pid}: invalid segmentation values"
                )

            if not set(np.unique(Y_class)).issubset(
                set(CLASS_NAMES)
            ):
                errors.append(
                    f"{pid}: invalid classification values"
                )

    if errors:
        print("VALIDATION FAILED")
        for error in errors[:100]:
            print(" -", error)
        raise RuntimeError(
            f"{len(errors)} validation errors."
        )

    print("Complete 500-patient patch dataset validation: PASS")


if RUN_FULL_EXTRACTION:
    validate_dataset()
else:
    print("Full validation skipped because extraction is disabled.")


In [ ]:
# Cell 19 — Split-level patch statistics

if RUN_FULL_EXTRACTION:

    summary = []

    for split_name, patients in split_patients.items():

        counts = Counter()
        total = 0

        for pid in patients:

            y_path = (
                PATCH_ROOT
                / split_name
                / pid
                / "Y_class.npy"
            )

            y = np.load(
                y_path,
                mmap_mode="r",
            )

            total += len(y)
            counts.update(y.tolist())

        summary.append({
            "split": split_name,
            "patients": len(patients),
            "patches": total,
            "background": counts[0],
            "displaced": counts[1],
            "non_displaced": counts[2],
            "buckle": counts[3],
            "segmental": counts[4],
        })

    split_summary = pd.DataFrame(summary)

    split_summary.to_csv(
        PATCH_ROOT / "split_patch_summary.csv",
        index=False,
    )

    print(split_summary)

else:
    print("Patch statistics skipped.")


In [ ]:
# Cell 20 — Save final extraction configuration

config = {
    "dataset": "RibFrac labelled 500-patient cohort",
    "source_counts": {
        "training_part_1": 300,
        "training_part_2": 120,
        "labelled_validation": 80,
    },
    "total_patients": 500,
    "patient_split": {
        "method": "patient-level random split",
        "train_patients": 350,
        "validation_patients": 75,
        "internal_test_patients": 75,
        "random_state": SEED,
    },
    "patch_size": list(PATCH_SIZE),
    "stride": list(STRIDE),
    "negative_to_positive_ratio": NEGATIVE_TO_POSITIVE_RATIO,
    "min_fracture_voxels": MIN_FRACTURE_VOXELS,
    "ambiguous_label_code": AMBIGUOUS_LABEL_CODE,
    "segmentation_classes": {
        "0": "background",
        "1": "valid fracture",
    },
    "classification_classes": {
        "0": "background",
        "1": "displaced",
        "2": "non-displaced",
        "3": "buckle",
        "4": "segmental",
    },
    "ambiguous_patch_policy": "exclude",
    "mixed_class_tie_policy": "exclude",
    "official_ribfrac_test_labels_used": False,
    "internal_test_is_labelled_holdout": True,
    "full_extraction_executed": RUN_FULL_EXTRACTION,
}

CONFIG_PATH = (
    PATCH_ROOT / "patch_extraction_config.json"
)

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("Saved:", CONFIG_PATH)


In [ ]:
# Cell 21 — Kaggle dataset metadata

metadata = {
    "title": "RibFrac Full 500-Patient Patch Dataset",
    "id": "waboke/ribfrac-patch-dataset-final",
    "licenses": [{"name": "CC-BY-4.0"}],
    "description": (
        "Patient-level 3D patch dataset generated from the "
        "500 labelled RibFrac patients: 300 Training Part 1, "
        "120 Training Part 2, and 80 labelled validation cases. "
        "Separate train, validation, and internal held-out test "
        "patient directories are provided for binary rib-fracture "
        "segmentation and five-class fracture classification."
    ),
}

METADATA_PATH = (
    PATCH_ROOT / "dataset-metadata.json"
)

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved:", METADATA_PATH)


In [ ]:
# Cell 22 — Final inventory

print("=" * 80)
print("NOTEBOOK 03 — FULL 500-PATIENT EXTRACTION")
print("=" * 80)

print("Train patients:", len(train_patients))
print("Validation patients:", len(validation_patients))
print("Internal test patients:", len(test_patients))
print("Total patients:", len(all_patients))

print("\nPatch root:")
print(PATCH_ROOT)

if RUN_FULL_EXTRACTION:
    print(
        "\nPatient manifests:",
        len(list(PATCH_ROOT.rglob("manifest.csv")))
    )
    print(
        "NumPy arrays:",
        len(list(PATCH_ROOT.rglob("*.npy")))
    )
else:
    print("\nFull extraction has NOT been executed.")

print("\nNotebook 03 preparation complete.")
